<a href="https://colab.research.google.com/github/khovan123/cropstate/blob/main/notebooks/cropstate_colab_github_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CROPSTATE Colab Workflow

Notebook này dùng cho workflow:

- Code clone từ GitHub
- Dataset đọc từ Google Drive: `MyDrive/CROPSTATE_DATASET`
- Knowledge base đọc từ Google Drive: `MyDrive/CROPSTATE_KNOWLEDGE_BASE`
- Training outputs lưu vào Google Drive: `MyDrive/CROPSTATE_RESULTS`

Trước khi chạy training, vào `Runtime > Change runtime type` và chọn GPU.

In [7]:
# Sửa REPO_URL thành GitHub repo thật của bạn.
REPO_URL = "https://github.com/khovan123/cropstate"
PROJECT_DIR = "/content/CROPSTATE_Full_Research_Package"

DATA_ROOT = "/content/drive/MyDrive/CROPSTATE_DATASET"
KNOWLEDGE_ROOT = "/content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE"
RESULTS_ROOT = "/content/drive/MyDrive/CROPSTATE_RESULTS"
OUTPUT_DIR = f"{RESULTS_ROOT}/vision_final"

KB_COMPLETE_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_complete.jsonl"
KB_NONRESTRICTED_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_nonrestricted.jsonl"
RETRIEVAL_SCENARIOS = "data/retrieval_scenarios.csv"

print("REPO_URL:", REPO_URL)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("KNOWLEDGE_ROOT:", KNOWLEDGE_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

REPO_URL: https://github.com/khovan123/cropstate
PROJECT_DIR: /content/CROPSTATE_Full_Research_Package
DATA_ROOT: /content/drive/MyDrive/CROPSTATE_DATASET
KNOWLEDGE_ROOT: /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE
RESULTS_ROOT: /content/drive/MyDrive/CROPSTATE_RESULTS


## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Clone hoặc update repo từ GitHub

In [4]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/CROPSTATE_Full_Research_Package


## 3. Cài dependencies

In [5]:
!pip install -r requirements.txt
!pip install -e .

Obtaining file:///content/CROPSTATE_Full_Research_Package
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cropstate-research (pyproject.toml) ... done
  Created wheel for cropstate-research: filename=cropstate_research-0.1.0-0.editable-py3-none-any.whl size=1383 sha256=26a109959c066a78dc139ca7b88ba7058128ee79a12c71f9b814a690a640d6d8
  Stored in directory: /tmp/pip-ephem-wheel-cache-5vsw8_o2/wheels/37/37/8f/89550734e79d9e7fc68e041f688e3bc8f62ff3b93e9d2c762c
Successfully built cropstate-research


## 4. Kiểm tra dataset và knowledge base trên Drive

In [8]:
!ls -lah "{DATA_ROOT}"
!ls -lah "{KNOWLEDGE_ROOT}"

total 24K
drwx------ 2 root root 4.0K Jul  5 09:28 01_establishment
drwx------ 2 root root 4.0K Jul  6 16:19 02_tillering
drwx------ 2 root root 4.0K Jul  6 16:19 03_stem_booting
drwx------ 2 root root 4.0K Jul  6 16:19 04_reproductive
drwx------ 2 root root 4.0K Jul  5 11:04 05_grain_filling
drwx------ 2 root root 4.0K Jul  5 11:04 06_ripening
total 30K
-rw------- 1 root root 8.0K Jul  5 21:23 CROPSTATE_KB_Evaluation_Report.docx
-rw------- 1 root root  22K Jul  5 21:22 CROPSTATE_Sample_Knowledge_Base.xlsx


Dataset nên có cấu trúc:

```text
CROPSTATE_DATASET/
  01_establishment/
  02_tillering/
  03_stem_booting/
  04_reproductive/
  05_grain_filling/
  06_ripening/
```

Knowledge base nên có `CROPSTATE_Knowledge_Base_Complete.xlsx` và folder `chunks/` chứa `rice_knowledge_complete.jsonl` hoặc `rice_knowledge_nonrestricted.jsonl`.

## 5. Build manifest pilot từ folder dataset

In [9]:
!PYTHONPATH=src python scripts/build_stage_manifest.py \
  --data-root "{DATA_ROOT}" \
  --output data/stage_folder_manifest.csv

Wrote 557 rows to data/stage_folder_manifest.csv
macro_stage
tillering        150
ripening         148
grain_filling    125
stem_booting      54
reproductive      46
establishment     34


## 6. Auto rename ảnh và tạo manifest từ folder stage

Cell này tạo `image_manifest_auto.csv` trong `CROPSTATE_DATASET` và `rename_mapping_auto.csv` để lưu mapping tên cũ -> tên mới.

Mặc định `RENAME = False` để preview. Sau khi kiểm tra đúng, đổi thành `RENAME = True` và chạy lại cell.

Split theo từng folder stage:

- 5 ảnh đầu: `validation`
- 5 ảnh tiếp theo: `test`
- còn lại: `train`


In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT_PATH = Path(DATA_ROOT)
RENAME = False  # preview trước; đổi thành True để rename thật và tạo image_manifest_auto.csv

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

STAGES = {
    "01_establishment": "S01 Establishment",
    "02_tillering": "S02 Tillering",
    "03_stem_booting": "S03 Stem/booting",
    "04_reproductive": "S04 Reproductive",
    "05_grain_filling": "S05 Grain filling",
    "06_ripening": "S06 Ripening",
}

rows = []
rename_rows = []
rename_plan_by_folder = {}

for folder_name, label in STAGES.items():
    folder = DATA_ROOT_PATH / folder_name
    if not folder.exists():
        print(f"Missing folder: {folder}")
        continue

    images = sorted(
        [
            p for p in folder.iterdir()
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTS
            and not p.name.startswith("__tmp_cropstate_rename_")
        ],
        key=lambda p: p.name.lower(),
    )

    planned = []
    for i, old_path in enumerate(images, start=1):
        ext = old_path.suffix.lower()
        new_name = f"img{i:03d}{ext}"
        new_path = folder / new_name
        temp_path = folder / f"__tmp_cropstate_rename_{i:05d}{ext}"

        if i <= 5:
            split = "validation"
        elif i <= 10:
            split = "test"
        else:
            split = "train"

        new_stem = Path(new_name).stem
        image_id = f"{folder_name}_{new_stem}"
        parent_image_id = new_stem

        rename_rows.append({
            "stage_folder": folder_name,
            "old_name": old_path.name,
            "new_name": new_name,
            "old_path": str(old_path),
            "new_path": str(new_path),
            "split": split,
            "final_label": label,
            "already_named": old_path.name == new_name,
        })

        rows.append({
            "image_id": image_id,
            "file_name": f"{folder_name}/{new_name}",
            "drive_url": "",
            "source_name": "CROPSTATE_DATASET",
            "source_url": "",
            "license": "user_provided",
            "parent_image_id": parent_image_id,
            "field_id": f"{folder_name}_{new_stem}",
            "capture_session": f"{folder_name}_{new_stem}",
            "capture_date": "",
            "region": "",
            "variety": "unknown",
            "annotator_1": "",
            "annotator_2": "",
            "final_label": label,
            "usable": True,
            "split": split,
            "review_status": "folder_label_auto",
        })

        planned.append((old_path, temp_path, new_path))

    rename_plan_by_folder[folder_name] = planned

manifest = pd.DataFrame(rows)
rename_map = pd.DataFrame(rename_rows)

manifest_path = DATA_ROOT_PATH / "image_manifest_auto.csv"
preview_map_path = DATA_ROOT_PATH / "rename_mapping_preview.csv"
rename_map_path = DATA_ROOT_PATH / "rename_mapping_auto.csv"

if not RENAME:
    rename_map.to_csv(preview_map_path, index=False)
    print("RENAME = False: chỉ tạo preview, CHƯA rename ảnh và CHƯA tạo image_manifest_auto.csv")
    print("Preview mapping:", preview_map_path)
else:
    for planned in rename_plan_by_folder.values():
        for old_path, temp_path, _ in planned:
            old_path.rename(temp_path)
        for _, temp_path, new_path in planned:
            temp_path.rename(new_path)

    missing_after_rename = [
        str(DATA_ROOT_PATH / file_name)
        for file_name in manifest["file_name"]
        if not (DATA_ROOT_PATH / file_name).exists()
    ]
    if missing_after_rename:
        raise FileNotFoundError("Missing files after rename:\n" + "\n".join(missing_after_rename[:20]))

    manifest.to_csv(manifest_path, index=False)
    rename_map.to_csv(rename_map_path, index=False)
    print("RENAME = True: đã rename ảnh và tạo manifest")
    print("Manifest:", manifest_path)
    print("Rename mapping:", rename_map_path)

print("Rows:", len(manifest))
print()
print("Labels:")
print(manifest["final_label"].value_counts())
print()
print("Splits:")
print(manifest["split"].value_counts())
print()
display(rename_map.head(20))


## 7. Convert manifest auto thành manifest training

Cell này đọc `image_manifest_auto.csv` vừa tạo ở Drive và ghi ra `data/image_manifest.csv` trong repo Colab. Đây là manifest chính dùng để audit/train.


In [ ]:
!PYTHONPATH=src python scripts/convert_image_manifest.py \
  --input "{DATA_ROOT}/image_manifest_auto.csv" \
  --data-root "{DATA_ROOT}" \
  --output data/image_manifest.csv


## 8. Convert image manifest từ knowledge base 

Cell này ưu tiên đọc sheet `Image_Manifest_Template` trong `CROPSTATE_KNOWLEDGE_BASE`. Nếu sheet chưa được điền hoặc không có row training hợp lệ, script sẽ tự quét `DATA_ROOT` theo các folder stage và tạo manifest từ label folder.

Nó ghi ra `data/image_manifest_from_knowledge.csv` để không ghi đè `data/image_manifest.csv` từ auto manifest.


In [ ]:
!PYTHONPATH=src python scripts/convert_image_manifest.py \
  --knowledge-root "{KNOWLEDGE_ROOT}" \
  --data-root "{DATA_ROOT}" \
  --output data/image_manifest_from_knowledge.csv


## 9. Convert knowledge chunks

In [ ]:
!PYTHONPATH=src python scripts/convert_knowledge_base.py \
  --knowledge-root "{KNOWLEDGE_ROOT}" \
  --output data/knowledge_chunks.jsonl


## 10. Audit complete knowledge corpus

Dùng corpus complete để xem coverage, restricted chunks và trạng thái production. Output lưu vào Drive results.

In [ ]:
!mkdir -p "{RESULTS_ROOT}/retrieval"
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_COMPLETE_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_complete.json"


## 11. Audit non-restricted knowledge corpus

Corpus này đã loại chunk có restricted action, phù hợp hơn cho retrieval pilot.

In [ ]:
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_NONRESTRICTED_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_nonrestricted.json"


## 12. Run retrieval mẫu

Ví dụ: water management cho stage tillering. Script sẽ tải multilingual embedding model lần đầu chạy.

In [ ]:
!PYTHONPATH=src python scripts/run_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --topic water_management \
  --stage tillering \
  --mode research \
  --top-k 5 \
  --output "{RESULTS_ROOT}/retrieval/water_tillering.json"


## 12b. Liệt kê chunk_id thật để tạo `data/retrieval_scenarios.csv`

Cell "Evaluate retrieval baselines" bên dưới cần file `data/retrieval_scenarios.csv` — đây là file bạn tự viết tay (ground truth đánh giá), không phải file do script tự sinh ra. Nếu file này chưa tồn tại, cell evaluate sẽ báo `FileNotFoundError`, đó là điều bình thường, không phải lỗi.

Cell này chỉ **liệt kê chunk_id thật** trong corpus của bạn theo từng topic/stage, để bạn có cơ sở chọn:

- `relevant_chunk_ids`: các chunk phù hợp giai đoạn đó (stage_compatibility ≥ 0.8);
- `incompatible_chunk_ids`: các chunk cùng topic nhưng không phù hợp giai đoạn đó (stage_compatibility = 0).

**Quan trọng:** danh sách này chỉ là gợi ý tự động dựa trên vector `stage_compatibility` sẵn có trong corpus. Không nên dùng thẳng làm ground truth khoa học cho paper, vì `stage_compatibility` cũng chính là tín hiệu mà bộ rerank dùng để xếp hạng — dùng lại nó làm nhãn đánh giá sẽ gây thiên lệch có lợi cho chính phương pháp đang được đánh giá (circular evaluation). Hãy dùng danh sách này làm điểm khởi đầu, sau đó tự rà soát/chỉnh sửa (hoặc nhờ người có chuyên môn nông nghiệp xác nhận) trước khi đưa vào `data/retrieval_scenarios.csv` thật.


In [ ]:
import sys
sys.path.insert(0, "src")

from cropstate.knowledge import load_knowledge_chunks
from cropstate.constants import STAGE_NAMES

chunks = load_knowledge_chunks(KB_NONRESTRICTED_CORPUS, mode="research")
print(f"Total chunks loaded: {len(chunks)}\n")

topics = sorted({c.topic for c in chunks})
for topic in topics:
    topic_chunks = [c for c in chunks if c.topic == topic]
    print(f"=== topic: {topic} ({len(topic_chunks)} chunks) ===")
    for stage_index, stage in enumerate(STAGE_NAMES):
        relevant = [c for c in topic_chunks if c.stage_compatibility[stage_index] >= 0.8]
        incompatible = [c for c in topic_chunks if c.stage_compatibility[stage_index] == 0.0]
        if not relevant and not incompatible:
            continue
        print(f"  [{stage}]")
        if relevant:
            print(f"    relevant_chunk_ids candidates ({len(relevant)}):")
            for c in relevant[:8]:
                print(f"      {c.chunk_id}  -  {c.text[:70]}...")
        if incompatible:
            ids_preview = "|".join(c.chunk_id for c in incompatible[:8])
            print(f"    incompatible_chunk_ids candidates ({len(incompatible)}): {ids_preview}")
    print()

print("Copy các chunk_id phù hợp vào data/retrieval_scenarios.csv, cột relevant_chunk_ids")
print("và incompatible_chunk_ids cách nhau bằng dấu '|', ví dụ: kb_water_01|kb_water_07")


## 13. Evaluate retrieval baselines

Chỉ chạy cell này khi đã có `data/retrieval_scenarios.csv`. Metrics gồm P@k, R@k, nDCG@k, SIRR@k cho ungated, hard top-1, fixed-soft, adaptive-soft và oracle.

In [ ]:
!PYTHONPATH=src python scripts/evaluate_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --scenarios "{RETRIEVAL_SCENARIOS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/retrieval_evaluation.json"


## 14. Audit manifest pilot từ folder

In [ ]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum


## 15. Audit manifest auto dùng để train

Đây là manifest được tạo từ cell rename và convert ở trên.

In [ ]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/image_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum


## 16. Repair manifest từ file thật đang tồn tại

Dùng cell này nếu audit báo `Missing image files > 0`. Cell này không rename ảnh nữa; nó scan các file ảnh thật đang tồn tại trong `CROPSTATE_DATASET`, tạo lại `image_manifest_auto.csv`, rồi bạn chạy lại cell convert/audit phía dưới.

Split theo từng folder stage:

- 5 ảnh đầu: `validation`
- 5 ảnh tiếp theo: `test`
- còn lại: `train`


In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT_PATH = Path(DATA_ROOT)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

STAGES = {
    "01_establishment": "S01 Establishment",
    "02_tillering": "S02 Tillering",
    "03_stem_booting": "S03 Stem/booting",
    "04_reproductive": "S04 Reproductive",
    "05_grain_filling": "S05 Grain filling",
    "06_ripening": "S06 Ripening",
}

rows = []
for folder_name, label in STAGES.items():
    folder = DATA_ROOT_PATH / folder_name
    images = sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS],
        key=lambda p: p.name.lower(),
    )
    for i, path in enumerate(images, start=1):
        if i <= 5:
            split = "validation"
        elif i <= 10:
            split = "test"
        else:
            split = "train"

        stem = path.stem
        rows.append({
            "image_id": f"{folder_name}_{stem}",
            "file_name": f"{folder_name}/{path.name}",
            "drive_url": "",
            "source_name": "CROPSTATE_DATASET",
            "source_url": "",
            "license": "user_provided",
            "parent_image_id": stem.split("_subset_overlap", 1)[0],
            "field_id": f"{folder_name}_{stem}",
            "capture_session": f"{folder_name}_{stem}",
            "capture_date": "",
            "region": "",
            "variety": "unknown",
            "annotator_1": "",
            "annotator_2": "",
            "final_label": label,
            "usable": True,
            "split": split,
            "review_status": "folder_label_auto_repaired",
        })

manifest = pd.DataFrame(rows)
manifest_path = DATA_ROOT_PATH / "image_manifest_auto.csv"
manifest.to_csv(manifest_path, index=False)

print("Repaired manifest:", manifest_path)
print("Rows:", len(manifest))
print()
print("Labels:")
print(manifest["final_label"].value_counts())
print()
print("Splits:")
print(manifest["split"].value_counts())


## 17. Convert repaired manifest thành manifest training

Chạy cell này sau cell repair ở trên.

In [ ]:
!PYTHONPATH=src python scripts/convert_image_manifest.py \
  --input "{DATA_ROOT}/image_manifest_auto.csv" \
  --data-root "{DATA_ROOT}" \
  --output data/image_manifest.csv


## 18. Audit lại manifest đã repair

Chỉ train nếu `Missing image files: 0`. Nếu duplicate còn xuất hiện, có thể vẫn train pilot, nhưng không nên dùng kết quả làm báo cáo khoa học trước khi xử lý duplicate.

In [ ]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/image_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum


## 19. Train lần đầu vào `vision_final`

Chỉ dùng một output folder duy nhất. Script tự tạo `OUTPUT_DIR/manifest.csv` từ các stage folder trong `DATA_ROOT`, rồi lưu checkpoint và test metrics vào cùng folder.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --output "{OUTPUT_DIR}"

## 20. Fine-tune tiếp trong cùng `vision_final`

Dùng cell này cho lần chạy tiếp theo. Checkpoint cũ được đọc từ `OUTPUT_DIR/best_checkpoint.pt`; nếu validation tốt hơn, checkpoint mới ghi đè vào đúng file đó.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{OUTPUT_DIR}/manifest.csv" \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --resume-checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --freeze-backbone-epochs 0 \
  --learning-rate 0.0001 \
  --backbone-learning-rate 0.00001 \
  --output "{OUTPUT_DIR}"

## 21. Xem output đã lưu trên Drive

Folder này chỉ nên có `best_checkpoint.pt`, `history.json`, `class_counts.json`, `manifest.csv`, và `test_metrics.json`.

In [ ]:
!ls -lah "{OUTPUT_DIR}"

## 22. Test một ảnh upload từ máy

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
print("Uploaded:", image_path)

In [ ]:
!PYTHONPATH=src python scripts/predict_image.py \
  --checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --image "{image_path}"

## 23. Novelty experiments — ordinal/focal loss + fixed-split multi-seed (GPU)

Các thí nghiệm tăng novelty cho paper. Tất cả retrain ResNet-18 trên **cùng một split cố định** với `vision_final` (`OUTPUT_DIR/manifest.csv`) để so sánh trực tiếp với baseline CE:

- **ordinal** (Tier A#2): loss = CE + λ·(E[stage] − stage_thật)² → phạt mạnh lỗi cách xa giai đoạn, giảm MASD và lỗi non-adjacent.
- **focal** (Tier C#6): giảm trọng số mẫu dễ, giúp lớp hiếm (reproductive).
- **seed 7 / seed 123** (Tier C#9): cùng split cố định, chỉ đổi seed → tách phương sai khởi tạo khỏi phương sai split.

**Trước khi chạy:**
1. Đã `git push` code mới lên GitHub: `scripts/train_vision.py` (thêm `--loss`), `src/cropstate/losses.py`, `scripts/experiments/`.
2. Đã chạy lại **cell 2 (git pull)** để Colab có code mới.
3. Đã chạy **cell 19** (train `vision_final`) để có split cố định.

Colab có GPU nên **không** cần `CROPSTATE_FORCE_CPU`. Mỗi lần train ~vài phút trên GPU.

In [ ]:
NOVELTY_DIR = f"{RESULTS_ROOT}/novelty"
FIXED_MANIFEST = f"{OUTPUT_DIR}/manifest.csv"  # split cố định từ baseline CE (cell 19)

import os
os.makedirs(NOVELTY_DIR, exist_ok=True)
assert os.path.exists(FIXED_MANIFEST), (
    f"Thiếu {FIXED_MANIFEST} — chạy cell 19 (train vision_final) trước để tạo split cố định."
)
print("Fixed split :", FIXED_MANIFEST)
print("Novelty dir :", NOVELTY_DIR)

In [ ]:
# Tier A#2 — ordinal loss (CE + λ·(E[stage] − stage_thật)²)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss ordinal \
  --output "{NOVELTY_DIR}/resnet18_ordinal"

In [ ]:
# Tier C#6 — focal loss (giúp lớp hiếm reproductive)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss focal \
  --output "{NOVELTY_DIR}/resnet18_focal"

In [ ]:
# Tier C#9 — fixed-split, seed 7
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 7 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed7"

In [ ]:
# Tier C#9 — fixed-split, seed 123
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 123 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed123"

## 24. So sánh biến thể novelty với baseline CE

Đọc `test_metrics.json` của từng biến thể (cùng split cố định) và in bảng: accuracy, macro-F1, MASD, ECE, recall lớp reproductive. Kỳ vọng: **ordinal** giảm MASD và lỗi non-adjacent; **focal** tăng recall reproductive; **seed7/seed123** cho dải phương sai khởi tạo trên split cố định.

In [ ]:
import json, os
import pandas as pd

variants = {
    "CE baseline (vision_final)": OUTPUT_DIR,
    "ordinal (A#2)":              f"{NOVELTY_DIR}/resnet18_ordinal",
    "focal (C#6)":                f"{NOVELTY_DIR}/resnet18_focal",
    "CE seed7 (C#9)":             f"{NOVELTY_DIR}/resnet18_ce_seed7",
    "CE seed123 (C#9)":           f"{NOVELTY_DIR}/resnet18_ce_seed123",
}
rows = []
for name, d in variants.items():
    p = os.path.join(d, "test_metrics.json")
    if not os.path.exists(p):
        print("bỏ qua (chưa train):", name)
        continue
    m = json.load(open(p))
    rows.append({
        "variant": name,
        "accuracy": round(m.get("accuracy", float("nan")), 3),
        "macro_f1": round(m.get("macro_f1", float("nan")), 3),
        "masd": round(m.get("masd", float("nan")), 3),
        "ece": round(m.get("ece", float("nan")), 3),
        "reproductive_recall": round((m.get("per_class_recall") or {}).get("reproductive", float("nan")), 3),
    })

df = pd.DataFrame(rows)
df.to_csv(f"{NOVELTY_DIR}/retrain_comparison.csv", index=False)
print(df.to_string(index=False))
print("\nĐã lưu:", f"{NOVELTY_DIR}/retrain_comparison.csv")

## 25. (Tuỳ chọn) Phân tích post-hoc trên Colab → lưu vào Drive

Export logits từ các checkpoint có sẵn rồi chạy calibration (temperature scaling, Tier A#3), temporal fusion (Tier B#5) và leakage audit (Tier A#1). Cần checkpoint `vision_final` (và nếu có `vision_small_cnn`, `vision_efficientnet_b0`) trong `RESULTS_ROOT`. Kết quả JSON lưu vào `RESULTS_ROOT/novelty` để kéo về.

In [ ]:
CKPTS = {
    "resnet18": f"{OUTPUT_DIR}/best_checkpoint.pt",
    "small_cnn": f"{RESULTS_ROOT}/vision_small_cnn/best_checkpoint.pt",
    "efficientnet_b0": f"{RESULTS_ROOT}/vision_efficientnet_b0/best_checkpoint.pt",
}
import os
ck = " ".join(f'--checkpoint {n}={p}' for n, p in CKPTS.items() if os.path.exists(p))
print("Checkpoints dùng được:", ck or "(chỉ resnet18)")

!PYTHONPATH=src python scripts/experiments/export_logits.py {ck} \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --output-dir "{NOVELTY_DIR}/logits"

# Calibration (temperature scaling) — torch-free, cắt ECE
!PYTHONPATH=src python scripts/experiments/calibration.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" \
  --output "{NOVELTY_DIR}/calibration.json"

# Temporal phenology fusion (Tier B#5)
!PYTHONPATH=src python scripts/experiments/temporal_fusion.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" --model resnet18 \
  --output "{NOVELTY_DIR}/temporal_fusion.json"

# Near-duplicate / leakage audit (Tier A#1)
!PYTHONPATH=src python scripts/experiments/leakage_audit.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" --hamming-threshold 10 \
  --output "{NOVELTY_DIR}/leakage_audit.json"

print("Xong. Kết quả trong", NOVELTY_DIR)

## Ghi chú

- GitHub chỉ lưu code và metadata nhỏ.
- Google Drive lưu dataset, knowledge base, checkpoint và kết quả training.
- Không commit các file `.pt` lên GitHub.